# 06 · Sites and terrain

Groups the competitions into flying sites and describes the terrain of every 1 km cell around them, from the Copernicus GLO-30 elevation model (30 m, global, free). Adds to `database/`:

| Table | One row per |
|---|---|
| `sites` | flying site: its box, competitions and flights |
| `event_sites` | competition, with its site |
| `site_chips` | 0.25° chip of a site (the box around one ERA5 grid point, used for the weather in the next step) |
| `cells` | 1 km cell (0.01°): elevation, slope, which way it faces, ridge or valley, local relief, and the slope and aspect shares used to compute sunlight on the ground at any hour |

The DEM tiles are saved in `dem/glo30/` in the project folder (about 30 MB per 1° tile) and reused by later steps for height above ground. Run notebook 05 first: the sites are drawn from where pilots actually flew.

Tiles and chips are cached, so **if Colab disconnects, Run All again** and it carries on.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
WORKERS = 2          # processes computing terrain features (Colab usually has 2 CPUs)
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive and get the code

In [ ]:
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

drive.mount("/content/drive")
auth.authenticate_user()
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
DEM_DIR = os.path.join(ROOT, "dem", "glo30")
assert os.path.isfile(os.path.join(DB, "flights.parquet")), "Run notebook 05 first: database/flights.parquet is missing"
print("Database in", DB)
print("DEM tiles go to", DEM_DIR)
subprocess.run(["pip", "install", "-q", "rasterio"], check=True)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

## 3. Sites

Groups the competitions by where they flew and prints the sites with the most flights. Takes seconds.

In [ ]:
run(f"{REPO}/scripts/build_sites.py", DB)

## 4. Terrain

Downloads the DEM tiles the sites need, then computes the features of every cell. Prints progress; expect roughly 20–40 minutes for the download and a similar time for the cells, depending on the number of sites.

In [ ]:
run(f"{REPO}/scripts/build_terrain.py", DB, "--dem", DEM_DIR, "--workers", str(WORKERS))

## 5. A first look

In [ ]:
import duckdb, pandas as pd

db = duckdb.connect()
for name in ["sites", "event_sites", "site_chips", "cells", "flights"]:
    db.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{DB}/{name}.parquet')")

db.sql("""
    SELECT site_id, name, country, n_events, n_flights, n_chips, first_date, last_date
    FROM sites ORDER BY n_flights DESC LIMIT 30
""").df()

In [ ]:
# The sites on a map (circle size: flights)
import folium

sites = db.sql("SELECT * FROM sites").df()
m = folium.Map(location=[20, 0], zoom_start=2, tiles="OpenTopoMap")
for s in sites.itertuples():
    folium.Rectangle([[s.south, s.west], [s.north, s.east]], color="#2c3e50", weight=1, fill=False).add_to(m)
    folium.CircleMarker([s.lat, s.lon], radius=3 + s.n_flights ** 0.5 / 4, color="#c0392b", weight=0, fill=True,
                        fill_opacity=0.6, tooltip=f"{s.name}: {s.n_events} competitions, {s.n_flights} flights").add_to(m)
m

In [ ]:
# One site's terrain: elevation and how far each cell sits above (red) or below (blue) its surroundings
import matplotlib.pyplot as plt
import numpy as np

SITE = sites.iloc[0].site_id
c = db.sql(f"""
    SELECT c.* FROM cells c JOIN site_chips s USING (chip_id) WHERE s.site_id = '{SITE}'
""").df()
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, col, cmap in [(ax[0], "elev_mean", "terrain"), (ax[1], "tpi_5km", "RdBu_r")]:
    sc = a.scatter(c.lon, c.lat, c=c[col], s=4, marker="s", cmap=cmap,
                   **({"vmin": -150, "vmax": 150} if col == "tpi_5km" else {}))
    a.set_title(f"{sites.iloc[0]['name']}: {col}")
    a.set_aspect(1 / np.cos(np.radians(c.lat.mean())))
    plt.colorbar(sc, ax=a, shrink=0.8)
plt.show()